# 01 — Ingestão, tipagem e qualidade dos dados

**Grupo 1 · Tema G2 — Anomalia de carga**

Pergunta do tema: *em que momentos a rede sai do comportamento normal de forma
sustentada, e não num pico isolado?*

Este notebook cobre o primeiro elo da cadeia: **carregar a telemetria KPM, tipá-la e
medir a qualidade da amostra antes de calcular qualquer indicador**. Nenhuma conclusão
sobre anomalia é tirada aqui — os sete achados abaixo definem o que é honesto afirmar
nos notebooks seguintes.

Fonte obrigatória (trilha offline): `code/datasets/kpm-ue-tp-sample/kpm.sqlite`,
run `ue-tp-20260804-174422` do lab `oai-cn-gnb-nonrt-nearrt` (OAI `nr-softmodem` em
RFSIM + agente E2 + FlexRIC). Telemetria sintética, sem dados pessoais.

In [1]:
import sys
from pathlib import Path

sys.path.append("..")          # code/ -> g2_lib.py
from g2_lib import *           # noqa: F403

RAIZ = Path("../..").resolve()
AMOSTRA = RAIZ / "code" / "datasets" / "kpm-ue-tp-sample"
DERIVED = RAIZ / "derived"
FIGURES = RAIZ / "figures"
DERIVED.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

print("raiz do pacote:", RAIZ.name)
print("fonte:", (AMOSTRA / "kpm.sqlite").relative_to(RAIZ))

raiz do pacote: _pr
fonte: code\datasets\kpm-ue-tp-sample\kpm.sqlite


## 1. Extract — o que existe no SQLite

Duas tabelas: `runs` (proveniência) e `kpm_samples` (as amostras, com as métricas
serializadas em `payload_json`).

In [2]:
import sqlite3
import pandas as pd

con = sqlite3.connect(AMOSTRA / "kpm.sqlite")
display(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con))
display(pd.read_sql("SELECT * FROM runs", con))
display(pd.read_sql(
    "SELECT phase, sample_index, ingested_at, substr(payload_json,1,70) AS payload FROM kpm_samples LIMIT 4", con))
con.close()

,name
0,kpm_samples
1,runs
2,sqlite_sequence


,run_id,created_at,use_case,notes
0,ue-tp-20260804-174422,2026-08-04T20:44:22.310716+00:00,ue-tp-load-anomaly,baseline lab OAI+RFSIM+FlexRIC


,phase,sample_index,ingested_at,payload
0,baseline,0,2026-08-04T20:44:22.310716+00:00,"{""DRB.RlcSduDelayDl"": 55.0, ""DRB.UEThpUl"": 4.4..."
1,baseline,1,2026-08-04T20:44:22.310716+00:00,"{""DRB.RlcSduDelayDl"": 218.0, ""DRB.UEThpUl"": 3...."
2,baseline,2,2026-08-04T20:44:22.310716+00:00,"{""DRB.RlcSduDelayDl"": 137.0, ""DRB.UEThpUl"": 3...."
3,baseline,3,2026-08-04T20:44:22.310716+00:00,"{""DRB.RlcSduDelayDl"": 171.0, ""DRB.UEThpUl"": 3...."


## 2. Transform — expandir o payload em colunas tipadas

`carregar_amostras()` faz o parse do JSON, converte para `float64` e ordena as fases
na ordem do experimento (baseline → stress → recovery).

In [3]:
amostras = carregar_amostras(AMOSTRA / "kpm.sqlite")
print(f"{len(amostras)} amostras · {amostras['run_id'].nunique()} run · {amostras['phase'].nunique()} fases")
display(amostras.head(3))
display(amostras.dtypes.to_frame("tipo"))

100 amostras · 1 run · 3 fases


,id,run_id,phase,sample_index,ingested_at,source_path,DRB.RlcSduDelayDl,DRB.UEThpUl,RRU.PrbTotUl
0,1,ue-tp-20260804-174422,baseline,0,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,55.0,4.46,2.0
1,2,ue-tp-20260804-174422,baseline,1,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,218.0,3.72,2.0
2,3,ue-tp-20260804-174422,baseline,2,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,137.0,3.72,2.0


,tipo
id,int64
run_id,str
phase,category
sample_index,int64
ingested_at,str
source_path,str
DRB.RlcSduDelayDl,float64
DRB.UEThpUl,float64
RRU.PrbTotUl,float64


## 3. Consulta 1 (SQL) — contagem por fase

Confere contra o `db_summary.json` distribuído com a amostra.

In [4]:
con = sqlite3.connect(AMOSTRA / "kpm.sqlite")
contagem = pd.read_sql(
    """
    SELECT phase, COUNT(*) AS n,
           COUNT(DISTINCT ingested_at) AS timestamps_distintos
    FROM kpm_samples GROUP BY phase ORDER BY phase
    """, con)
con.close()
display(contagem)

oficial = pd.read_json(AMOSTRA / "db_summary.json")
print("confere com db_summary.json:",
      dict(zip(contagem.phase, contagem.n)) == dict(zip(oficial.phase, oficial["count"])))

,phase,n,timestamps_distintos
0,baseline,20,1
1,recovery,20,1
2,stress,60,1


confere com db_summary.json: True


## 4. Consulta 2 (pandas) — perfil das três métricas por fase

Primeira leitura substantiva: as fases se separam? E em que direção?

In [5]:
perfil = (amostras.groupby("phase", observed=True)[FEATURES]
          .agg(["median", "mean", "min", "max"]).round(2))
display(perfil)

for f in FEATURES:
    med = amostras.groupby("phase", observed=True)[f].median()
    print(f"{f:22s} ({UNIDADES[f]:>4s})  baseline={med['baseline']:>12,.2f}   "
          f"stress={med['stress']:>12,.2f}   recovery={med['recovery']:>12,.2f}")

DRB.RlcSduDelayDl                         DRB.UEThpUl            \
                    median    mean     min     max      median      mean   
phase                                                                      
baseline               0.0   55.25    0.00  218.00        3.72      3.72   
stress               158.9  161.98  133.74  264.75    80023.68  78383.75   
recovery               0.0   78.85    0.00  470.00        3.72   8619.34   

                           RRU.PrbTotUl                    
            min        max       median   mean  min   max  
phase                                                      
baseline   3.00       4.46          2.0   2.00  2.0   2.0  
stress    15.16   82823.86         99.0  97.33  2.0  99.0  
recovery   3.00  172316.90          2.0   2.95  2.0  21.0

DRB.RlcSduDelayDl      (  us)  baseline=        0.00   stress=      158.90   recovery=        0.00
DRB.UEThpUl            (kbps)  baseline=        3.72   stress=   80,023.68   recovery=        3.72
RRU.PrbTotUl           (   %)  baseline=        2.00   stress=       99.00   recovery=        2.00


> **Leitura.** A fase de stress eleva as três métricas ao mesmo tempo: PRB de 2 % para
> 99 %, vazão de 3,72 kbps para ~80 Mbps e atraso de 0 para ~159 µs. A anomalia deste
> lab é de **carga**, não de falha — por isso a vazão *sobe* em vez de degradar.

## 5. Controle de qualidade — os sete achados

A rubrica dá 2,0 pontos para aquisição, preparação e qualidade dos dados. Cada item
abaixo é calculado, não afirmado, e vai para o README e para o slide.

In [6]:
import json
qc = relatorio_qc(amostras)
print(json.dumps(qc, indent=2, ensure_ascii=False, default=str))

{
  "linhas": 100,
  "runs": [
    "ue-tp-20260804-174422"
  ],
  "amostras_por_fase": {
    "baseline": 20,
    "stress": 60,
    "recovery": 20
  },
  "timestamps_distintos": 3,
  "delay_zero_por_fase": {
    "baseline": {
      "zeros": 11,
      "total": 20
    },
    "stress": {
      "zeros": 0,
      "total": 60
    },
    "recovery": {
      "zeros": 11,
      "total": 20
    }
  },
  "maior_repeticao_de_payload": 10,
  "linhas_duplicadas": 18,
  "fracao_nula": {
    "DRB.RlcSduDelayDl": 0.0,
    "DRB.UEThpUl": 0.0,
    "RRU.PrbTotUl": 0.0
  },
  "DRB.UEThpDl_presente": false,
  "coluna_ue_id_presente": false,
  "fracao_stress": 0.6,
  "colunas": [
    "phase",
    "sample_index",
    "DRB.RlcSduDelayDl",
    "DRB.UEThpUl",
    "RRU.PrbTotUl"
  ],
  "gerado_em": "2026-08-29T13:31:39.262555+00:00"
}


### Achado 1 — não há timestamp por amostra

As 100 linhas compartilham apenas **3** valores distintos de `ingested_at`, um por
arquivo de fase. O eixo "tempo" real é o `sample_index` dentro da fase.

**Consequência:** nenhuma métrica em unidade de tempo (segundos de anomalia,
taxa por minuto) é defensável. Medimos duração em número de amostras consecutivas.

In [7]:
display(amostras.assign(f=amostras["phase"].astype(str))
        .groupby("ingested_at")
        .agg(n=("id", "size"), fases=("f", lambda s: ", ".join(sorted(set(s))))))

,n,fases
ingested_at,,
2026-08-04T20:44:22.310716+00:00,20,baseline
2026-08-04T20:44:22.532550+00:00,60,stress
2026-08-04T20:44:22.670212+00:00,20,recovery


### Achados 2 e 3 — atraso zero e duplicatas exatas

`DRB.RlcSduDelayDl = 0` aparece em 11 das 20 amostras de baseline e em 11 das 20 de
recovery. Zero de atraso de RLC não é fisicamente plausível: é **métrica não reportada
na Indication**, gravada como 0 pelo parser do lab.

A mesma tupla `(0 ; 3,72 ; 2,0)` se repete 10 vezes em cada fase calma — repetição do
mesmo valor de contador, não amostragem independente.

**Consequência:** tratada como valor válido, ela puxa a mediana de baseline para 0 e faz
com que *qualquer* atraso não-nulo pareça anômalo. É a correção V2 do notebook 02.

In [8]:
delay = "DRB.RlcSduDelayDl"
zeros = (amostras.assign(zero=amostras[delay].eq(0))
         .groupby("phase", observed=True)["zero"].agg(zeros="sum", total="size"))
zeros["pct"] = (100 * zeros.zeros / zeros.total).round(1)
display(zeros)

duplicatas = (amostras.groupby(["phase"] + FEATURES, observed=True).size()
              .reset_index(name="repeticoes").query("repeticoes > 1")
              .sort_values("repeticoes", ascending=False))
display(duplicatas)

,zeros,total,pct
phase,,,
baseline,11,20,55.0
stress,0,60,0.0
recovery,11,20,55.0


,phase,DRB.RlcSduDelayDl,DRB.UEThpUl,RRU.PrbTotUl,repeticoes
1,baseline,0.0,3.72,2.0,10
72,recovery,0.0,3.72,2.0,10


### Achado 4 — o baseline é degenerado

O MAD das três features no baseline é **zero**: a fase calma é praticamente constante.
Como `escala = max(MAD × 1,4826 ; 1,0)`, o **piso** assume o papel do desvio robusto e
o "z-score" degenera em **desvio absoluto na unidade nativa** de cada métrica.

**Consequência:** os escores ficam incomparáveis entre features — o de vazão, em kbps,
chega a dezenas de milhares enquanto o de PRB, em %, fica na casa das dezenas. É o que
motiva a saturação do indicador ISC no notebook 02.

In [9]:
diag = amostras[amostras.phase == "baseline"][FEATURES].agg(["median", "std"]).T
diag["mad"] = [(amostras[amostras.phase == "baseline"][f]
                - amostras[amostras.phase == "baseline"][f].median()).abs().median() for f in FEATURES]
diag["escala_efetiva"] = (diag["mad"] * ESCALA_MAD).clip(lower=MAD_FLOOR)
diag["unidade"] = [UNIDADES[f] for f in FEATURES]
display(diag.round(3))

,median,std,mad,escala_efetiva,unidade
DRB.RlcSduDelayDl,0.00,76.039,0.0,1.0,us
DRB.UEThpUl,3.72,0.237,0.0,1.0,kbps
RRU.PrbTotUl,2.00,0.000,0.0,1.0,%


### Achados 5, 6 e 7 — coluna ausente, UE único, desbalanceamento

- O README da amostra cita `DRB.UEThpDl`, mas a coluna **não existe** neste dataset.
- Não há `ue_id` no payload: é **um único UE**. Qualquer agregação "de célula" é
  didática, não estatística.
- **60 %** das amostras são de stress. Toda taxa precisa ser reportada **por fase**;
  uma taxa global ficaria inflada.

In [10]:
con = sqlite3.connect(AMOSTRA / "kpm.sqlite")
chaves = set()
for (p,) in con.execute("SELECT payload_json FROM kpm_samples"):
    chaves |= set(json.loads(p))
con.close()
print("chaves presentes no payload:", sorted(chaves))
print("DRB.UEThpDl presente?", "DRB.UEThpDl" in chaves)
print("ue_id presente?", "ue_id" in chaves)
print("\nfracao de amostras por fase:")
display((amostras["phase"].value_counts(normalize=True) * 100).round(1).to_frame("%"))

chaves presentes no payload: ['DRB.RlcSduDelayDl', 'DRB.UEThpUl', 'RRU.PrbTotUl']
DRB.UEThpDl presente? False
ue_id presente? False

fracao de amostras por fase:


,%
phase,
stress,60.0
baseline,20.0
recovery,20.0


## 6. Load — exportar a camada tratada

Duas saídas em `derived/`: a tabela de features (`kpm_features.csv`) e o relatório de
qualidade (`etl_qc.json`). A fonte `kpm.sqlite` **não é alterada** — todo tratamento
acontece a jusante e é auditável.

In [11]:
features = amostras[["run_id", "phase", "sample_index", "ingested_at"] + FEATURES]
features.to_csv(DERIVED / "kpm_features.csv", index=False)
salvar_json(qc, DERIVED / "etl_qc.json")

print("gravado:", (DERIVED / "kpm_features.csv").relative_to(RAIZ), f"({len(features)} linhas)")
print("gravado:", (DERIVED / "etl_qc.json").relative_to(RAIZ))
display(features.groupby("phase", observed=True)[FEATURES].median().round(2))

gravado:

 derived\kpm_features.csv (100 linhas)
gravado: derived\etl_qc.json


,DRB.RlcSduDelayDl,DRB.UEThpUl,RRU.PrbTotUl
phase,,,
baseline,0.0,3.72,2.0
stress,158.9,80023.68,99.0
recovery,0.0,3.72,2.0


## O que este notebook estabelece

| # | Achado | Efeito na análise |
|---|--------|-------------------|
| 1 | 3 timestamps para 100 amostras | Sem métrica em unidade de tempo |
| 2 | Atraso = 0 em 11/20 nas fases calmas | Motiva a variante V2 |
| 3 | 10 duplicatas exatas por fase calma | Baseline menos informativo que n=20 sugere |
| 4 | MAD zero → piso vira a escala | Motiva a saturação do ISC |
| 5 | `DRB.UEThpDl` ausente | Só 3 features disponíveis |
| 6 | UE único, sem `ue_id` | Agregação de célula é didática |
| 7 | 60 % das amostras em stress | Todo indicador reportado por fase |

**Próximo:** `02_kpis_g2.ipynb` — os dois indicadores e as visualizações.